In [1]:
%matplotlib inline
# Cell 1 — parameters (match step 1)
LAT       = 16.8167
LON       = -2.9833
RADIUS_KM = 100
LEVEL     = '06'
EPSILON   = 0.001
ZERO_FRACTION_THRESHOLD = 0.20   # zero-inflation cutoff; matches step 3

In [2]:
# Cell 2 — imports and connection
import sys
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

sys.path.insert(0, '../../../..')
from scripts.shared.db_utils import db_connect
import scripts.shared.db_utils as _dbu

conn = db_connect()
print('Connected:', conn.execute('SELECT current_database()').fetchone())

ROOT  = Path(_dbu.__file__).parent.parent.parent
OUT   = ROOT / 'output' / 'edop' / 'areas'
OUT.mkdir(parents=True, exist_ok=True)

TABLE = f'public.basin{LEVEL}'
VIEW  = f'public.v_basin{LEVEL}_persist_rev1'

Connected: ('cedop',)


In [3]:
# Cell 3 — re-run buffer resolver → weighted basin set + hybas_id list
RADIUS_M = RADIUS_KM * 1000

resolver_sql = f"""
WITH pt AS (
    SELECT ST_SetSRID(ST_MakePoint({LON}, {LAT}), 4326)::geography AS pt_geog
),
buf AS (
    SELECT ST_Buffer(pt_geog, {RADIUS_M}) AS buf_geog,
           ST_Area(ST_Buffer(pt_geog, {RADIUS_M})) AS buf_area_m2
    FROM pt
),
candidates AS (
    SELECT b.hybas_id,
           ST_Area(ST_Intersection(b.geog, buf.buf_geog)) AS overlap_m2,
           buf.buf_area_m2
    FROM {TABLE} b, buf
    WHERE ST_Intersects(b.geog, buf.buf_geog)
)
SELECT hybas_id, overlap_m2 / buf_area_m2 AS weight
FROM candidates
WHERE overlap_m2 / buf_area_m2 >= {EPSILON}
ORDER BY weight DESC
"""

basin_set  = pd.read_sql(resolver_sql, conn).set_index('hybas_id')
hybas_ids  = basin_set.index.tolist()
ids_clause = ', '.join(str(h) for h in hybas_ids)

print(f'Basin set: {len(hybas_ids)} basins')
print(basin_set.round(4))

Basin set: 9 basins
              weight
hybas_id            
1.060042e+09  0.2771
1.060565e+09  0.1738
1.060041e+09  0.1628
1.060565e+09  0.1367
1.060552e+09  0.1060
1.060552e+09  0.0876
1.060044e+09  0.0255
1.060583e+09  0.0181
1.060583e+09  0.0124


In [4]:
# Cell 4 — load catalog; build variable lists and metadata DataFrame
#
# Exclusions:
#   gdp_avg, human_dev_idx — dropped (coarse economic variables, per design)
#   coast_flag, endorheic  — emitted as raw flags (0/1), not scored
#   deferred               — Band T time-dependent variables
#   dominance_class        — pnv_shares composite

CAT_PATH = ROOT / 'documentation' / 'EDOPS_variable_catalog_v0.3.tsv'
cat = pd.read_csv(CAT_PATH, sep='\t')

impl = cat[
    (cat['status'] == 'implemented') &
    (~cat['position_method'].isin(['deferred', 'dominance_class']))
].copy()

impl = impl[~impl['basin08_col_s'].fillna('').str.contains('\\.\\.')]

EXCLUDE_KEYS = {'gdp_avg', 'human_dev_idx'}
FLAG_KEYS    = {'coast_flag', 'endorheic'}

level_suffix = 'L6' if LEVEL == '06' else 'L8'

def _val(v):
    if v is None:
        return None
    try:
        if np.isnan(float(v)):
            return None
    except (TypeError, ValueError):
        pass
    s = str(v).strip()
    return s if s and s.lower() != 'nan' else None

def _zf(v):
    """Return float zero_fraction or None if missing/non-numeric."""
    try:
        f = float(v)
        return f if not np.isnan(f) else None
    except (TypeError, ValueError):
        return None

var_rows = []
for _, row in impl.iterrows():
    for su, api_col, db_col in [
        ('s', 'api_key_s', 'basin08_col_s'),
        ('u', 'api_key_u', 'basin08_col_u'),
    ]:
        api_key    = _val(row.get(api_col))
        db_colname = _val(row.get(db_col))
        if not api_key or not db_colname:
            continue
        if api_key in EXCLUDE_KEYS:
            continue
        if api_key in FLAG_KEYS:
            kind = 'flag'
        elif row['position_method'] == 'rarity_rank':
            kind = 'categorical'
        else:
            kind = 'continuous'
        zf_col = f'zero_fraction_{su}_{level_suffix}'
        var_rows.append({
            'api_key':         api_key,
            'schema_key':      row['schema_key'],
            'su':              su,
            'db_col':          db_colname,
            'kind':            kind,
            'band':            row['band'],
            'position_method': row['position_method'],
            'typology_cluster':row.get('typology_cluster'),
            'zero_fraction':   _zf(row.get(zf_col)),
        })

meta_df = pd.DataFrame(var_rows).set_index('api_key')

cont  = meta_df[meta_df['kind'] == 'continuous']
cats  = meta_df[meta_df['kind'] == 'categorical']
flags = meta_df[meta_df['kind'] == 'flag']

zf_above = meta_df[
    (meta_df['kind'] == 'continuous') &
    (meta_df['zero_fraction'].notna()) &
    (meta_df['zero_fraction'] >= ZERO_FRACTION_THRESHOLD)
]

print(f'Implemented variables: {len(impl)} schema rows')
print(f'  continuous : {len(cont)}')
print(f'  categorical: {len(cats)}')
print(f'  flag       : {len(flags)}')
print(f'  excluded   : {EXCLUDE_KEYS}')
print(f'\nZero-inflated continuous vars (zf >= {ZERO_FRACTION_THRESHOLD}, level={LEVEL}):')
if len(zf_above):
    for k, r in zf_above.iterrows():
        print(f'  {k:35s}  zero_fraction={r["zero_fraction"]:.3f}')
else:
    print('  (none)')

Implemented variables: 46 schema rows
  continuous : 41
  categorical: 11
  flag       : 2
  excluded   : {'human_dev_idx', 'gdp_avg'}

Zero-inflated continuous vars (zf >= 0.2, level=06):
  karst                                zero_fraction=0.696
  karst_upstream                       zero_fraction=0.612
  permafrost_extent                    zero_fraction=0.766
  wet_pct_grp1                         zero_fraction=0.331
  wet_pct_grp1_upstream                zero_fraction=0.358
  wet_pct_grp2                         zero_fraction=0.519
  wet_pct_grp2_upstream                zero_fraction=0.503
  cropland_extent                      zero_fraction=0.429
  cropland_extent_upstream             zero_fraction=0.406
  pasture_extent                       zero_fraction=0.329
  pasture_extent_upstream              zero_fraction=0.313
  dist_sink                            zero_fraction=0.334


In [5]:
# Cell 5 — raw values from view for our basin set
#
# SELECT * then drop geometry; view handles temperature ÷10 and lookup joins.
# Mask NoData sentinel (-9999) to NaN after fetching.

raw_sql = f"""
SELECT * FROM {VIEW}
WHERE hybas_id IN ({ids_clause})
"""

raw_all = pd.read_sql(raw_sql, conn).set_index('hybas_id')
raw_all = raw_all.drop(columns=['geom'], errors='ignore')

# Replace -9999 NoData sentinel with NaN across all numeric columns
num_cols = raw_all.select_dtypes(include='number').columns
raw_all[num_cols] = raw_all[num_cols].replace(-9999, np.nan)

# Attach weights
raw_all = basin_set[['weight']].join(raw_all)

# Subset to just our variable api_keys + weight
keep_cols = ['weight'] + [k for k in meta_df.index if k in raw_all.columns]
missing_from_view = [k for k in meta_df.index if k not in raw_all.columns]

raw_df = raw_all[keep_cols].copy()

print(f'Raw values: {len(raw_df)} basins × {len(raw_df.columns)-1} variable columns')
if missing_from_view:
    print(f'NOT found in view ({len(missing_from_view)}): {missing_from_view}')
print()
display(raw_df)

,weight,elev_min,elev_max,slope_avg,slope_upstream,stream_gradient,lith_class,karst,karst_upstream,permafrost_extent,...,cropland_extent,cropland_extent_upstream,pasture_extent,pasture_extent_upstream,pop_density,human_footprint_09,human_footprint_09_upstream,dist_sink,endorheic,coast_flag
hybas_id,,,,,,,,,,,,,,,,,,,,,
1.060042e+09,0.277141,237,492,5,3,5,Unconsolidated Sediments (SU),0,0,0,...,0,0,0,0,1.895,15,18,0.0,2,0
1.060565e+09,0.173836,256,339,2,13,5,Unconsolidated Sediments (SU),0,1,0,...,4,10,60,39,18.700,65,88,2530.7,0,0
1.060041e+09,0.162803,252,325,6,6,4,Unconsolidated Sediments (SU),0,0,0,...,0,0,0,0,0.396,10,10,0.0,2,0
1.060565e+09,0.136735,241,517,5,5,12,Unconsolidated Sediments (SU),0,0,0,...,1,2,37,38,8.505,49,49,2530.8,0,0
1.060552e+09,0.105959,257,279,1,12,4,Unconsolidated Sediments (SU),0,1,0,...,0,9,28,39,26.705,63,86,2331.1,0,0
1.060552e+09,0.087555,253,403,5,4,11,Unconsolidated Sediments (SU),0,0,0,...,1,1,50,50,3.141,50,50,2331.3,0,0
1.060044e+09,0.025482,250,407,2,2,5,Unconsolidated Sediments (SU),0,0,0,...,0,0,6,6,0.430,14,14,0.0,2,0
1.060583e+09,0.018053,257,1019,9,9,16,Unconsolidated Sediments (SU),0,0,0,...,11,11,48,48,15.876,64,64,2625.6,0,0
1.060583e+09,0.012435,256,440,4,13,8,Unconsolidated Sediments (SU),0,1,0,...,5,10,41,39,32.727,81,89,2625.8,0,0


In [6]:
# Cell 6 — position scores for continuous variables via PERCENT_RANK()
#
# One SQL scans the full basin table once, computing PERCENT_RANK() for every
# continuous variable, then filters to our hybas_ids.
# Recipes from catalog position_method:
#   percentile     → PERCENT_RANK() OVER (ORDER BY raw_val NULLS LAST)
#   log_percentile → PERCENT_RANK() OVER (ORDER BY LN(1 + GREATEST(0, raw_val)) NULLS LAST)
# NoData (-9999) and NULL → outer CASE returns NULL (not a high rank).
#
# Zero-aware variant (activated when zero_fraction >= ZERO_FRACTION_THRESHOLD):
#   Zeros score 0.0 explicitly. Non-zeros are ranked only within the positive
#   subpopulation via PARTITION BY, so the zero pile does not compress their range.

def rank_expr(db_col, method, zero_fraction=None):
    nodata = f"({db_col} = -9999 OR {db_col} IS NULL)"
    if method == 'log_percentile':
        val = f"LN(1.0 + GREATEST(0.0, {db_col}::float))"
    else:
        val = f"{db_col}::float"

    if zero_fraction is not None and zero_fraction >= ZERO_FRACTION_THRESHOLD:
        # Zero-aware: PARTITION BY sign splits the window so non-zeros are ranked
        # only among themselves (0–100 within the positive subpopulation).
        pos_val = (f"CASE WHEN {nodata} OR {db_col} <= 0 THEN NULL "
                   f"ELSE {val} END")
        return (
            f"CASE WHEN {nodata} THEN NULL "
            f"WHEN {db_col} = 0 THEN 0.0 "
            f"ELSE PERCENT_RANK() OVER ("
            f"PARTITION BY CASE WHEN {db_col} > 0 THEN 1 ELSE 0 END "
            f"ORDER BY {pos_val} NULLS LAST) * 100 END"
        )
    else:
        order_expr = f"CASE WHEN {nodata} THEN NULL ELSE {val} END"
        return (
            f"CASE WHEN {nodata} THEN NULL "
            f"ELSE PERCENT_RANK() OVER (ORDER BY {order_expr} NULLS LAST) * 100 END"
        )

cont_vars = meta_df[meta_df['kind'] == 'continuous']

select_parts = ['hybas_id']
alias_map    = {}

for api_key, row in cont_vars.iterrows():
    alias = f'pos_{api_key}'
    zf = row.get('zero_fraction')
    select_parts.append(
        f"{rank_expr(row['db_col'], row['position_method'], zf)} AS {alias}"
    )
    alias_map[alias] = api_key

rank_sql = f"""
WITH ranked AS (
    SELECT {', '.join(select_parts)}
    FROM {TABLE}
)
SELECT * FROM ranked
WHERE hybas_id IN ({ids_clause})
"""

pos_raw = pd.read_sql(rank_sql, conn).set_index('hybas_id')
pos_df  = pos_raw.rename(columns=alias_map)

print(f'Position scores: {len(pos_df)} basins × {len(pos_df.columns)} continuous variables')
print(f'Score range across non-null cells: {pos_df.min().min():.1f} – {pos_df.max().max():.1f}')
null_count = pos_df.isna().sum().sum()
print(f'Null scores (NoData raw values): {null_count}')

zf_vars_in = cont_vars[
    cont_vars['zero_fraction'].notna() &
    (cont_vars['zero_fraction'] >= ZERO_FRACTION_THRESHOLD)
].index.tolist()
zf_vars_in = [v for v in zf_vars_in if v in pos_df.columns]
if zf_vars_in:
    print(f'\nZero-aware scored variables ({len(zf_vars_in)}):')
    display(pos_df[zf_vars_in].round(1))

print('\nAll continuous scores:')
display(pos_df.round(1))

,elev_min,elev_max,slope_avg,slope_upstream,stream_gradient,karst,karst_upstream,permafrost_extent,discharge_yr,discharge_min,...,aridity,aridity_upstream,cropland_extent,cropland_extent_upstream,pasture_extent,pasture_extent_upstream,pop_density,human_footprint_09,human_footprint_09_upstream,dist_sink
hybas_id,,,,,,,,,,,,,,,,,,,,,
1.060042e+09,62.0,31.8,15.4,6.7,3.7,0.0,0.0,0.0,46.3,34.9,...,8.3,7.5,0.0,0.0,0.0,0.0,41.0,26.5,27.3,0.0
1.060565e+09,62.4,33.2,15.4,12.3,12.2,0.0,0.0,0.0,54.8,47.8,...,10.8,9.9,0.0,10.0,63.8,64.3,56.5,49.5,48.5,83.7
1.060044e+09,63.4,25.9,6.2,3.9,3.7,0.0,0.0,0.0,24.5,0.0,...,8.3,7.5,0.0,0.0,22.5,20.3,28.2,25.7,24.9,0.0
1.060041e+09,63.6,19.7,18.6,14.8,2.4,0.0,0.0,0.0,18.0,0.0,...,6.3,5.8,0.0,0.0,0.0,0.0,27.6,22.3,21.7,0.0
1.060552e+09,63.8,25.6,15.4,9.1,10.9,0.0,0.0,0.0,33.4,33.6,...,12.8,11.5,0.0,0.0,74.6,75.8,46.0,50.3,49.3,80.2
1.060565e+09,64.1,20.7,6.2,30.3,3.7,0.0,0.0,0.0,86.0,89.9,...,13.8,37.8,23.6,45.6,82.1,65.3,66.5,60.1,73.6,83.7
1.060583e+09,64.1,28.2,12.3,30.3,7.3,0.0,0.0,0.0,83.8,0.0,...,14.7,39.2,28.2,45.6,67.4,65.3,73.8,69.3,74.1,85.0
1.060552e+09,64.3,16.3,2.8,28.2,2.4,0.0,0.0,0.0,84.9,89.3,...,10.8,36.5,0.0,42.9,55.5,65.3,71.4,58.9,72.5,80.2
1.060583e+09,64.3,55.4,26.0,21.8,17.9,0.0,0.0,0.0,69.2,59.8,...,18.4,16.8,46.9,48.5,72.8,73.9,64.5,59.5,59.2,85.0


In [7]:
# Cell 7 — class identity for categorical variables; raw values for flag variables
#
# Categoricals: emit the class label (from the view, which joins lookup tables) and
# the integer class ID (from the raw table). Step 3 computes class frequencies
# downstream; Step 2 only identifies which class each basin belongs to.
#
# Flags (coast_flag, endorheic): emit the raw integer directly from raw_df.
# These are boolean/ordinal identifiers, not ranked or scored here.

cat_vars  = meta_df[meta_df['kind'] == 'categorical']
flag_vars = meta_df[meta_df['kind'] == 'flag']

class_label_rows = {}   # api_key → Series of text labels (from view)
class_id_rows    = {}   # api_key → Series of integer IDs (from raw table)

for api_key, row in cat_vars.iterrows():
    db_col = row['db_col']

    # Integer class ID from raw table
    id_sql = f"""
        SELECT hybas_id, {db_col} AS class_id
        FROM {TABLE}
        WHERE hybas_id IN ({ids_clause})
    """
    id_series = pd.read_sql(id_sql, conn).set_index('hybas_id')['class_id']
    class_id_rows[api_key] = id_series

    # Text label from the view (already in raw_df — view joins lookup tables)
    if api_key in raw_df.columns:
        class_label_rows[api_key] = raw_df[api_key]

class_label_df = pd.DataFrame(class_label_rows, index=raw_df.index)
class_id_df    = pd.DataFrame(class_id_rows,    index=raw_df.index)

# Flags: raw integer values from raw_df
flag_cols = [k for k in flag_vars.index if k in raw_df.columns]
flag_df   = raw_df[flag_cols].copy()

print(f'Categoricals: {len(class_label_df.columns)} variables')
print(f'Flags:        {len(flag_df.columns)} variables  {flag_cols}')
print()
print('--- class labels ---')
display(class_label_df)
print('--- class IDs ---')
display(class_id_df)
print('--- flags ---')
display(flag_df)

,endorheic,coast_flag
hybas_id,,
1.060042e+09,2,0
1.060565e+09,0,0
1.060041e+09,2,0
1.060565e+09,0,0
1.060552e+09,0,0
1.060552e+09,0,0
1.060044e+09,2,0
1.060583e+09,0,0
1.060583e+09,0,0


In [8]:
# Cell 8 — assemble Step 2 output matrix and sanity report
#
# Three DataFrames produced by this notebook (Step 3 inputs):
#
#   raw_df        — weight + raw values for all variables (from view)
#   matrix_df     — the typed Step 2 matrix Step 3 will aggregate:
#                     continuous vars → position score 0–100
#                     categorical vars → class label (text)
#                     flag vars        → raw integer (0/1/2)
#   class_id_df   — integer class IDs for categoricals (supplement to labels)
#   meta_df       — index=api_key; kind, band, position_method, typology_cluster

var_cols = [c for c in raw_df.columns if c != 'weight']

# Continuous scores from pos_df; categoricals as labels; flags as raw int
matrix_df = pd.concat([
    pos_df,
    class_label_df,
    flag_df,
], axis=1).reindex(columns=var_cols)

n_basins = len(raw_df)
n_vars   = len(var_cols)

cont_cols  = [c for c in var_cols if c in pos_df.columns]
cat_cols   = [c for c in var_cols if c in class_label_df.columns]
flag_c     = [c for c in var_cols if c in flag_df.columns]

print(f'Matrix: {n_basins} basins × {n_vars} variables')
print(f'  continuous scores : {len(cont_cols)}')
print(f'  categorical labels: {len(cat_cols)}')
print(f'  flags (raw int)   : {len(flag_c)}')
print()

# Null audit on continuous scores
score_nulls = matrix_df[cont_cols].isna().sum()
null_score_vars = score_nulls[score_nulls > 0]
if len(null_score_vars):
    print('Null continuous scores (NoData in source):')
    print(null_score_vars.to_string())
else:
    print('✓ No null continuous scores')

null_labels = matrix_df[cat_cols].isna().sum()
null_label_vars = null_labels[null_labels > 0]
if len(null_label_vars):
    print(f'\nNull categorical labels (orphaned FK or missing data):')
    print(null_label_vars.to_string())
else:
    print('✓ No null categorical labels')

print()
print('--- continuous scores (first 6 vars) ---')
display(matrix_df[cont_cols[:6]].round(1))
print('--- categorical labels ---')
display(matrix_df[cat_cols])
print('--- flags ---')
display(matrix_df[flag_c])

,endorheic,coast_flag
hybas_id,,
1.060042e+09,2,0
1.060565e+09,0,0
1.060044e+09,2,0
1.060041e+09,2,0
1.060552e+09,0,0
1.060565e+09,0,0
1.060583e+09,0,0
1.060552e+09,0,0
1.060583e+09,0,0


In [9]:
# Cell 9 — persist Step 2 outputs to output/edop/areas/
raw_df.to_csv(OUT / 'step2_raw.tsv', sep='\t', float_format='%.4f')
matrix_df.to_csv(OUT / 'step2_matrix.tsv', sep='\t')
class_id_df.to_csv(OUT / 'step2_class_ids.tsv', sep='\t')
meta_df.to_csv(OUT / 'step2_meta.tsv', sep='\t')

print(f'Saved to {OUT}:')
print(f'  step2_raw.tsv       ({raw_df.shape[0]} basins × {raw_df.shape[1]} cols)')
print(f'  step2_matrix.tsv    ({matrix_df.shape[0]} basins × {matrix_df.shape[1]} cols)')
print(f'  step2_class_ids.tsv ({class_id_df.shape[0]} basins × {class_id_df.shape[1]} cols)')
print(f'  step2_meta.tsv      ({meta_df.shape[0]} variables)')

Saved to /Users/karlg/Documents/repos/_edops/output/edop/areas:
  step2_raw.tsv       (9 basins × 55 cols)
  step2_matrix.tsv    (9 basins × 54 cols)
  step2_class_ids.tsv (9 basins × 11 cols)
  step2_meta.tsv      (54 variables)


In [10]:
# Cell 9 — close connection
conn.close()
print('Done.')

Done.
